# Feature → MITRE ATT&CK Mapping: Validation

The **core contribution**: an empirically-validated mapping from NFStream flow-statistical
features to ATT&CK techniques, intended to be reliable enough to serve as ground truth.

A `feature → class → TTP` link is accepted only when **three independent lines of evidence agree**:
1. **XAI importance** — the model relies on it (SHAP on the deployed binary model + an auxiliary
   3-class model + EBM glass-box + LIME).
2. **Statistical discrimination** — it separates the class from benign, with a direction and effect size.
3. **Literature** — prior work explains the causal link (Phase 4, see references table).

Classes → techniques: **C2 beaconing** → T1071 / T1071.001 / T1573 ; **Exfiltration** → T1041 / T1048.002.


## 0. Imports & data

In [ ]:
import os, numpy as np, pandas as pd, joblib
from scipy.stats import mannwhitneyu
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler

DATA = next((p for p in ["../data/training_dataset.csv","data/training_dataset.csv",
                         os.path.expanduser("~/dev/data/training_dataset.csv")] if os.path.exists(p)), None)
MAPPER = next((p for p in ["../models/mapper","models/mapper",
                           os.path.expanduser("~/dev/models/mapper")] if os.path.exists(p)), None)
df = pd.read_csv(DATA); mapper = joblib.load(MAPPER)
FEATS = mapper["REALTIME_SAFE_FEATURES"]

Xr = df[FEATS].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan)
Xr = Xr.fillna(Xr.median())
scaler = mapper["scaler_rt"]
X  = pd.DataFrame(scaler.transform(Xr), columns=FEATS)
cls = df["class"]                              # benign / c2_beaconing / exfil
print(f"{len(df)} flows | {len(FEATS)} features | classes {cls.value_counts().to_dict()}")


## Phase 2 — Statistical discrimination (per class vs benign)
Mann–Whitney U + rank-biserial effect size + direction (median comparison). This establishes *which* features separate each class from benign and *in which direction*.

In [ ]:
ben = Xr[cls=="benign"]
def signature(c):
    mal = Xr[cls==c]; rows=[]
    for f in FEATS:
        a,b = mal[f].values, ben[f].values
        u,p = mannwhitneyu(a,b,alternative="two-sided")
        rbc = 2*u/(len(a)*len(b)) - 1
        rows.append((f, abs(rbc), "HIGH" if np.median(a)>np.median(b) else "LOW", p))
    return (pd.DataFrame(rows, columns=["feature","effect","direction","p"])
              .query("p < 0.01").sort_values("effect", ascending=False))

stat_sig = {c: signature(c) for c in ["c2_beaconing","exfil"]}
for c,s in stat_sig.items():
    print(f"\n=== {c} vs benign (top 8) ==="); print(s.head(8).to_string(index=False))


## Phase 1a — SHAP on the **deployed binary model** (xgb_rt), stratified by class
Confirms the features the *actually-deployed* model uses to flag each class, with signed direction (+ pushes toward malicious).

In [ ]:
import shap
xgb = mapper["xgb_rt"]
idx = np.random.RandomState(42).choice(len(X), min(3000,len(X)), replace=False)
sv  = shap.TreeExplainer(xgb).shap_values(X.iloc[idx])
ci  = cls.values[idx]

def shap_sig(c):
    s = sv[ci==c]
    return pd.DataFrame({"feature":FEATS,
                         "mean_abs_shap":np.abs(s).mean(0),
                         "signed":s.mean(0)}).sort_values("mean_abs_shap",ascending=False)
shap_bin = {c: shap_sig(c) for c in ["c2_beaconing","exfil"]}
for c,s in shap_bin.items():
    print(f"\n=== SHAP(binary) {c} (top 8) ===")
    for _,r in s.head(8).iterrows():
        print(f"  {'mal+' if r.signed>0 else 'ben-'}  {r.mean_abs_shap:.3f}  {r.feature}")


## Phase 1b — Auxiliary 3-class model + SHAP (benign / C2 / exfil)
Direct per-class attribution as a cross-check on the stratified binary view.

In [ ]:
from xgboost import XGBClassifier
y3 = cls.map({"benign":0,"c2_beaconing":1,"exfil":2})
xgb3 = XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.8,
                     objective="multi:softprob", num_class=3, eval_metric="mlogloss",
                     n_jobs=-1, random_state=42)
xgb3.fit(X, y3)
sv3 = shap.TreeExplainer(xgb3).shap_values(X.iloc[idx])   # list[3] of (n,feat)
LAB = {1:"c2_beaconing", 2:"exfil"}
shap_3c = {}
for k,name in LAB.items():
    mean_abs = pd.Series(np.abs(sv3[:,:,k]).mean(0), index=FEATS)
    shap_3c[name] = mean_abs.sort_values(ascending=False)
    print(f"\n=== SHAP(3-class) {name} (top 8) ==="); print(shap_3c[name].head(8).to_string())


## Phase 3 — Consensus: feature → class assignment
A feature is assigned to a class only if it is **top-k in the statistical test AND in ≥1 SHAP view**, with a consistent direction. This consensus set is what feeds the `feature → class → TTP` map.

In [ ]:
TOPK = 12
def topset(series_or_df, col=None):
    s = series_or_df if col is None else series_or_df.set_index("feature")[col]
    return set(s.sort_values(ascending=False).head(TOPK).index)

consensus = {}
for c in ["c2_beaconing","exfil"]:
    stat_top = set(stat_sig[c].head(TOPK)["feature"])
    shapb_top = topset(shap_bin[c].set_index("feature")["mean_abs_shap"])
    shap3_top = topset(shap_3c[c])
    agreed = stat_top & (shapb_top | shap3_top)
    dirmap = dict(zip(stat_sig[c]["feature"], stat_sig[c]["direction"]))
    consensus[c] = sorted(((f, dirmap[f]) for f in agreed), key=lambda x:x[0])
    print(f"\n=== {c}: consensus features (stat AND shap) ===")
    for f,d in consensus[c]: print(f"  {d:4}  {f}")


## TODO — remaining phases
- **EBM glass-box + LIME** triangulation (add as further evidence columns; EBM shape functions confirm direction exactly).
- **Phase 3 stability**: bootstrap the SHAP/stat rankings → rank confidence intervals; cross-model Spearman (RF vs XGB vs EBM).
- **Phase 4 — literature grounding**: cite prior work for each consensus `feature → class` link (references table).
- **Phase 5 — build map**: rewrite `services/translator/feature_mitre_map.py` as `feature → class → TTP` with evidence metadata; update `translate()` + alert schema.
- **Phase 6 — end-to-end validation**: run XAI→translate() on held-out flows, measure TTP-assignment precision/recall vs `attack_type`.
